# In Silico Validation and Dynamic Modeling of Maize Single-Cell RNA

### **Overview**
This notebook implements an end-to-end, reproducible pipeline for analyzing cellular dynamics in *Zea mays* (Maize) using single-cell RNA sequencing (scRNA-seq) data. By quantifying both nascent and mature transcripts, this workflow models the future state of individual plant cells, allowing us to identify key driver genes responsible for developmental transitions and cellular trajectories.

### **The Technique: RNA Velocity**
Traditional scRNA-seq provides a static "snapshot" of cellular states. To move beyond static profiling, this pipeline utilizes **RNA Velocity**—a powerful computational technique that estimates the rate of gene expression change by analyzing the balance between **unspliced (pre-mRNA)** and **spliced (mature mRNA)** transcripts.
* A high ratio of unspliced to spliced mRNA indicates that a gene is being actively upregulated (turned on).
* A low ratio indicates that a gene is being downregulated (turned off).
By calculating these dynamic rates across thousands of genes, we can project the future transcriptomic state of each cell, effectively mapping the directional flow of cellular differentiation and development.

### **The Technology Stack**
This workflow leverages a modern, high-performance bioinformatics stack designed for rapid and memory-efficient single-cell analysis:
* **Alevin-fry:** A fast, memory-frugal tool used for pseudoalignment and quantification. It utilizes the Unspliced-Spliced-Ambiguous (USA) mode to efficiently parse raw `FASTQ` reads against the reference index, generating the distinct spliced and unspliced count matrices required for velocity analysis.
* **Scanpy:** A scalable Python toolkit for analyzing single-cell gene expression data. It is used here for critical preprocessing steps: quality control, cell normalization, log-transformation, and identifying highly variable genes.
* **scVelo:** The premier Python package for RNA velocity analysis. It computes the first and second moments to smooth technical noise, solves the dynamical equations of transcription, and computes the high-dimensional velocity vectors.
* **Google Colab (GPU-enabled):** Provides the containerized computational environment to run the pipeline uniformly without local hardware constraints.

### **The Analytical Approach**
The pipeline is structured into a logical, step-by-step progression:
1. **Environment & Data Setup:** Automates the installation of system dependencies (`sra-tools`, `alevin-fry`) and Python libraries, followed by downloading the Maize reference genome and raw sequencing data.
2. **Splici-aware Quantification:** Uses `alevin-fry` to align reads and collate them by cell barcode, explicitly distinguishing between unspliced and spliced molecules.
3. **Data Preprocessing:** Loads the count matrices into an `AnnData` object, filtering for highly variable genes and computing neighborhood moments to stabilize expression values.
4. **Velocity Inference & Projection:** Calculates the RNA velocity graph and projects the dynamic vectors onto a 2D UMAP embedding. Streamline plots are generated to visualize the developmental trajectories of the Maize cells.
5. **Driver Gene Discovery:** Extracts the dynamic velocity scores to computationally identify which specific genes are actively driving the observed developmental transitions. The highest-scoring genes are exported to a CSV for downstream biological validation.

---
*Note: Ensure that the Colab runtime is set to GPU (Runtime > Change runtime type > GPU) for optimal processing speed during the dimensionality reduction and graphing steps.*

In [ ]:
# Create the directory
!mkdir -p maize_reference

# Change into the directory (use % for cd in Colab)
%cd maize_reference

# Download and unzip the FASTA
!wget -c https://ftp.ensemblgenomes.ebi.ac.uk/pub/plants/current/fasta/zea_mays/dna/Zea_mays.Zm-B73-REFERENCE-NAM-5.0.dna.toplevel.fa.gz
!gunzip Zea_mays.Zm-B73-REFERENCE-NAM-5.0.dna.toplevel.fa.gz

# Download and unzip the GTF
# Download the exact GTF file (version 62)
!wget -c https://ftp.ensemblgenomes.ebi.ac.uk/pub/plants/current/gtf/zea_mays/Zea_mays.Zm-B73-REFERENCE-NAM-5.0.62.gtf.gz

# Decompress the file
!gunzip Zea_mays.Zm-B73-REFERENCE-NAM-5.0.62.gtf.gz

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!conda install -c conda-forge -c bioconda pyroe bedtools -y

In [ ]:
import os
# Delete the conflicting Colab graphing variable
if 'MPLBACKEND' in os.environ:
    del os.environ['MPLBACKEND']



In [ ]:
# 3. Build the splici reference
# The '91' represents the standard read length for 10x Genomics single-cell RNA-seq
!pyroe make-splici \
    Zea_mays.Zm-B73-REFERENCE-NAM-5.0.dna.toplevel.fa \
    Zea_mays.Zm-B73-REFERENCE-NAM-5.0.62.gtf \
    91 \
    maize_splici_out

In [ ]:
!conda install -c bioconda salmon -y

In [ ]:
# -t specifies our input fasta
# -i specifies the name of the output folder for the index
# -p 2 tells Colab to use 2 CPU cores to speed it up
!salmon index -t maize_splici_out/splici_fl86.fa -i maize_splici_index -p 2

In [ ]:
!conda install -c bioconda sra-tools -y

In [ ]:
# 1. Create a folder to hold your raw experimental data
!mkdir -p /content/maize_fastq
%cd /content/maize_fastq

# 2. Download the first run (SRR7989735)
!fasterq-dump SRR7989735 --split-files --progress

# 3. Download the second run (SRR7989736)
!fasterq-dump SRR7989736 --split-files --progress

In [ ]:
# Print the first 4 lines of the Read 1 FASTQ file
!head -n 4 SRR7989735_1.fastq

In [ ]:
!salmon alevin -l ISR \
  -1 SRR7989735_1.fastq \
  -2 SRR7989735_2.fastq \
  --bc-geometry 1[1-6] \
  --umi-geometry 1[7-16] \
  --read-geometry 2[1-end] \
  -i /content/maize_reference/maize_splici_index \
  -p 2 \
  -o alevin_out_7989735 \
  --tgMap /content/maize_reference/maize_splici_out/splici_fl86_t2g_3col.tsv \
  --rad

In [ ]:
import json

# Open the metadata file created by Salmon Alevin
file_path = 'alevin_out_7989735/aux_info/meta_info.json'

try:
    with open(file_path) as f:
        meta = json.load(f)

    processed = meta.get('num_processed', 0)
    mapped = meta.get('num_mapped', 0)

    print(f" Total reads processed: {processed:,}")
    print(f"Total reads successfully mapped: {mapped:,}")

    if processed > 0:
        rate = (mapped / processed) * 100
        print(f" Mapping Rate: {rate:.2f}%")
except FileNotFoundError:
    print("Could not find the meta_info.json file. Make sure you are in the correct directory!")

In [ ]:
!salmon alevin -l ISR \
  -1 SRR7989736_1.fastq \
  -2 SRR7989736_2.fastq \
  --bc-geometry 1[1-6] \
  --umi-geometry 1[7-16] \
  --read-geometry 2[1-end] \
  -i /content/maize_reference/maize_splici_index \
  -p 2 \
  -o alevin_out_7989736 \
  --tgMap /content/maize_reference/maize_splici_out/splici_fl86_t2g_3col.tsv \
  --rad

In [ ]:
!conda install -c bioconda alevin-fry -y

In [ ]:
# -d both: Look at reads mapping to both strands (standard for this protocol)
# -i: The input folder from our Salmon Alevin run
# -o: The output folder for the permit list
# --force-cells 500: Extract the top 500 cell barcodes
!alevin-fry generate-permit-list -d both -i alevin_out_7989735 -o permit_7989735 --force-cells 500

In [ ]:
!alevin-fry generate-permit-list -d both -i alevin_out_7989736 -o permit_7989736 --force-cells 500

In [ ]:
# 1. Collate the reads by cell barcode
!alevin-fry collate -t 2 -i permit_7989735 -r alevin_out_7989735

# 2. Quantify the spliced and unspliced transcripts
!alevin-fry quant -t 2 \
  -i permit_7989735 \
  -o quant_7989735 \
  --tg-map /content/maize_reference/maize_splici_out/splici_fl86_t2g_3col.tsv \
  --resolution cr-like \
  --use-mtx

In [ ]:
# 1. Collate the reads by cell barcode
!alevin-fry collate -t 2 -i permit_7989736 -r alevin_out_7989736

# 2. Quantify the spliced and unspliced transcripts
!alevin-fry quant -t 2 \
  -i permit_7989736 \
  -o quant_7989736 \
  --tg-map /content/maize_reference/maize_splici_out/splici_fl86_t2g_3col.tsv \
  --resolution cr-like \
  --use-mtx

In [ ]:
!pip install scvelo anndata

In [ ]:
%pip install pyroe scanpy scvelo anndata

In [ ]:
%%bash

# Write the fixed Python script
cat << 'EOF' > make_h5ad.py
import pyroe
import anndata as ad

print("Loading AR Stage data...")
adata_AR = pyroe.load_fry('quant_7989735')
adata_AR.obs['condition'] = 'AR_Stage'

print("Loading PMC Stage data...")
adata_PMC = pyroe.load_fry('quant_7989736')
adata_PMC.obs['condition'] = 'PMC_Stage'

print("Merging datasets...")
# index_unique='-' forces the barcodes to be unique immediately during the merge
adata = ad.concat([adata_AR, adata_PMC], label='batch', index_unique='-')

# THE FIX: Remove the duplicate "barcodes" name so H5AD doesn't crash
adata.obs.index.name = None
if 'barcodes' in adata.obs.columns:
    del adata.obs['barcodes']

print("Saving to H5AD file...")
adata.write_h5ad('maize_velocity_data.h5ad')
print("🎉 SUCCESS: Saved maize_velocity_data.h5ad!")
EOF

# Run the script
python make_h5ad.py

In [ ]:
import sys
!{sys.executable} -m pip install scanpy scvelo matplotlib -q
print("Installation forced into the current active kernel!")

In [ ]:
import scanpy as sc
import scvelo as scv
import matplotlib.pyplot as plt

# 1. Load the universal file using Scanpy instead!
adata = sc.read_h5ad('maize_velocity_data.h5ad')

print("--- Data Successfully Loaded ---")
print(adata)
print("\nCalculating Quality Control metrics...")

# 2. Calculate how many genes and total counts are in each cell
sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)

# 3. Draw a Violin plot to visualize the cell quality
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts'],
             jitter=0.4, multi_panel=True, groupby='condition')

In [ ]:
# 1. Copy the main matrix (.X) into a new layer called 'spliced' for scVelo
adata.layers['spliced'] = adata.X.copy()

print(f"Cells before filtering: {adata.n_obs}")

# 2. Filter out cells that have fewer than 200 genes (removes the empty droplets)
sc.pp.filter_cells(adata, min_genes=200)

# 3. Filter out extremely rare genes (keeps only genes found in at least 3 cells)
sc.pp.filter_genes(adata, min_cells=3)

print(f"Cells after filtering: {adata.n_obs}")

# Let's see the cleaned up object!
print("\n--- Cleaned Single-Cell Object ---")
print(adata)

In [ ]:
# 1. Let's look at the ratio of spliced vs unspliced RNA!
print("Calculating Spliced/Unspliced proportions...")
scv.pl.proportions(adata, groupby='condition')

# 3. Compute the neighborhood graph and moments (crucial for velocity)
print("Computing moments...")
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

print("\n--- Preprocessing Complete! ---")
print(adata)

In [ ]:
import scanpy as sc
import scvelo as scv

print("Step 1: Normalizing (Scanpy)...")
sc.pp.normalize_total(adata)

print("Step 2: Log transformation (Scanpy)...")
# THIS was the culprit! Using Scanpy's log1p instead.
sc.pp.log1p(adata)

print("Step 3: Finding Highly Variable Genes (Scanpy)...")
sc.pp.highly_variable_genes(adata, n_top_genes=2000)

print("Step 4: Computing moments (scVelo)...")
# This groups similar cells together to smooth out the noise for velocity
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

print("\n--- Preprocessing Complete! ---")
print(adata)

In [ ]:
import scanpy as sc
import scvelo as scv

print("1. Reloading fresh data...")
adata = sc.read_h5ad('maize_velocity_data.h5ad')
adata.layers['spliced'] = adata.X.copy()

print("2. Filtering empty droplets & rare genes...")
sc.pp.filter_cells(adata, min_genes=200) # Leaves us with our 552 real cells!
sc.pp.filter_genes(adata, min_cells=3)   # Light filter so we keep our genes!

print("3. Normalizing and Log-transforming...")
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

print("4. Finding Highly Variable Genes...")
sc.pp.highly_variable_genes(adata, n_top_genes=2000)

print("5. Computing PCA and Neighbors (Scanpy)...")
# This is the modern, warning-free way to group similar cells
sc.pp.pca(adata)
sc.pp.neighbors(adata, n_pcs=30, n_neighbors=30)

print("6. Computing moments (scVelo)...")
# Calculates the spliced/unspliced ratios for our grouped cells
scv.pp.moments(adata, n_pcs=None, n_neighbors=None)

print("\n--- Fresh Preprocessing Complete! ---")
print(adata)

In [ ]:
import scanpy as sc
import scvelo as scv
import matplotlib.pyplot as plt

print("1. Computing the 2D UMAP for visualization...")
sc.tl.umap(adata)

print("2. Calculating RNA Velocity (Stochastic mode)...")
# This computes the actual vector/direction each cell is moving
scv.tl.velocity(adata, mode='stochastic')

print("3. Building the Velocity Graph...")
# This translates the math into arrows we can project onto our UMAP
scv.tl.velocity_graph(adata)

print("\n--- Velocity Calculation Complete! ---")

# 4. Plotting the results!
print("Drawing the beautiful Velocity Stream plot...")
scv.pl.velocity_embedding_stream(adata, basis='umap', color='condition',
                                 title='Maize Anther Development Velocity',
                                 legend_loc='right margin')

In [ ]:
print("--- Proving the data is real! ---")

# Let's find the absolute highest number in the entire spliced table
max_value = df_spliced.max().max()
print(f"The highest single count in the table is: {max_value}")

# Let's add up every single number in the entire table
total_molecules = df_spliced.sum().sum()
print(f"Total RNA molecules counted in the whole table: {total_molecules}")

In [ ]:
import scvelo as scv
import pandas as pd

# 1. Estimate RNA velocity kinetics
scv.tl.velocity_graph(adata)

# 2. Identify the top genes driving the velocity (transition)
scv.tl.rank_velocity_genes(adata, groupby='condition', n_genes=10)

# 3. Extract the top 10 genes for the PMC stage transition as a DataFrame
top_genes = pd.DataFrame(adata.uns['rank_velocity_genes']['names'])
print("Top Velocity Driver Genes:")
print(top_genes.head(10))

# 4. Visualize the Phase Portraits for the top 3 driver genes
# These plots show the spliced vs unspliced relationship
top_3_genes = top_genes['PMC_Stage'].iloc[:3].tolist()
scv.pl.scatter(adata, basis=top_3_genes, ncols=3, frameon=False, title='Velocity Phase Portraits')


In [ ]:
import scvelo as scv

# 1. Recover dynamics (Required for latent time)
# This estimates transcription, splicing, and degradation rates
scv.tl.recover_dynamics(adata, n_jobs=2)

# 2. Compute Latent Time
scv.tl.latent_time(adata)

# 3. Plot the Heatmap
# Using the driver genes identified in the previous step
genes_to_plot = top_genes['PMC_Stage'].iloc[:10].tolist()

scv.pl.heatmap(adata,
               var_names=genes_to_plot,
               sortby='latent_time',
               col_color='condition',
               n_convolve=100,
               colorbar=True)

In [ ]:
# Save the final results to a new file for your paper
adata.write_h5ad('maize_final_velocity_results.h5ad')

# Optional: Export the driver gene list to a CSV for your supplemental data
top_genes.to_csv('top_velocity_drivers_maize.csv', index=False)

print("Files saved successfully. Your research data is now ready for writing.")

In [ ]:
import pandas as pd

# 1. Extract velocity values for all genes
# This represents the 'speed' of each gene's induction or repression
velocity_values = adata.layers['velocity']

# 2. Get the index of our top driver genes
# We focus on the PMC_Stage as it's the target of the transition
top_genes_list = top_genes['PMC_Stage'].iloc[:10].tolist()

# 3. Create a summary table with the average velocity score for each gene
velocity_summary = []
for gene in top_genes_list:
    gene_idx = adata.var_names.get_loc(gene)
    avg_velocity = adata.layers['velocity'][:, gene_idx].mean()
    velocity_summary.append({'Gene Symbol': gene, 'Average Velocity Score': avg_velocity})

# 4. Convert to DataFrame and display
df_velocity = pd.DataFrame(velocity_summary)
print("Table: Dynamic Velocity Scores of Key Driver Genes")
print(df_velocity.sort_values(by='Average Velocity Score', ascending=False))

# 5. Save this table for your paper's 'Results' section
df_velocity.to_csv('maize_driver_velocity_scores.csv', index=False)